# Comprehensive Strategy Analysis -- Liquidity Hedge Protocol v2

**SOL/USDC Corridor Derivative for Concentrated Liquidity Impermanent-Loss Hedging**

This notebook evaluates nine investment strategies using one year of real SOL/USDC market data,
comparing the protocol's two-part premium model against alternative hedging approaches.

---

## Part I: Setup & Market Data

### 1.1 Imports and Configuration

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from numpy.polynomial.hermite import hermgauss
from scipy.stats import norm
import requests
import struct
import base64
import time
import os
import warnings
from datetime import datetime, timedelta, timezone

warnings.filterwarnings('ignore')

# Ensure data directory exists
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'notebooks-v2', 'data')
os.makedirs(DATA_DIR, exist_ok=True)
# Also try relative
os.makedirs('data', exist_ok=True)
DATA_DIR = 'data'

# --- Protocol constants ---
PPM = 1_000_000
BPS = 10_000
N_LIQUIDITY = 10_000       # liquidity units
# NOTE: N_LIQUIDITY is the liquidity parameter L (Orca units), not dollar notional.
# Position value V0 = cl_value_vec(S0, L, p_l, p_u) varies with S0 and width.
# At S0=$84: V0 ≈ $4533 (±5%), $8975 (±10%). All returns are computed as % of V0.
T_WEEK = 7.0 / 365.0       # one week in years
MARKUP = 1.10               # 10% markup over fair value
ALPHA = 0.40                # 40% upfront fraction
BOND_APY = 0.12             # 12% risk-free benchmark
JITO_APY = 0.07             # 7% jitoSOL staking yield
SOL_FRACTION = 0.48         # SOL fraction in CL position (approx)
PERP_FUNDING_APY = 0.80     # Jupiter perp borrow rate ~80% APY (realistic, medium utilization)
PERP_FEE_BPS = 8             # 8 bps per side (open + close)
U_MAX_BPS = 3000            # 30% max utilization
PROTOCOL_FEE_BPS = 150      # 1.5% protocol fee

# ── Realistic competitor costs ──
IV_PREMIUM = 0.25              # implied vol premium over realized (25 pts typical for SOL)
OPTION_SPREAD_PCT = 0.10       # 10% bid-ask spread on options
PERP_PRICE_IMPACT_BPS = 3     # ~0.03% price impact per trade
TX_COST_SOL = 0.001            # ~$0.08 per transaction (priority fee)

# Width configurations
WIDTHS = {
    '5pct': {
        'width': 0.05,
        'barrier_pct': 0.95,
        'severity_ppm': 345_000,
        'fee_share_bps': 2500,
        'daily_fee': 0.0065,
        'label': '+/-5%',
    },
    '10pct': {
        'width': 0.10,
        'barrier_pct': 0.90,
        'severity_ppm': 420_000,
        'fee_share_bps': 2000,
        'daily_fee': 0.0045,
        'label': '+/-10%',
    },
}

# API keys
BIRDEYE_KEY = 'ed577a4a6a4f480fa659b4f18673e4b1'
HELIUS_RPC = 'https://mainnet.helius-rpc.com/?api-key=2ef5fdd0-5c3b-4ae1-a2fc-e12b3fd605e7'
SOL_MINT = 'So11111111111111111111111111111111111111112'
WHIRLPOOL_ACCOUNT = 'Czfq3xZZDmsdGdUyrNLtRhGc47cXcZtLG4crryfu44zE'

print("Configuration loaded successfully.")
print(f"  MARKUP = {MARKUP}x, ALPHA = {ALPHA}, T_WEEK = {T_WEEK:.5f} years")
print(f"  Widths: {[w['label'] for w in WIDTHS.values()]}")
# ── IV/RV-Adaptive Markup ──
MARKUP_FLOOR_VALUES = [1.03, 1.05, 1.07]
DEFAULT_MARKUP_FLOOR = 1.05

IV_RV_SCENARIOS = {
    'constant_1.25': lambda sigma7d, sigma30d, stress: 1.25,
    'regime_switching': lambda sigma7d, sigma30d, stress: (
        1.40 if stress or (sigma7d / max(sigma30d, 0.01) > 1.3) else 1.15
    ),
    'today_snapshot': None,  # filled from live Bybit IV in cell 19
}

def adaptive_markup(iv_rv_ratio, floor=DEFAULT_MARKUP_FLOOR):
    return max(floor, iv_rv_ratio)


### 1.2 Fetch 1-Year Daily SOL/USDC from Birdeye

In [ ]:
def fetch_birdeye_ohlcv(mint, api_key, days_back=365, chunk_days=90):
    """Fetch daily OHLCV from Birdeye in 90-day chunks."""
    url = 'https://public-api.birdeye.so/defi/ohlcv'
    headers = {'X-API-KEY': api_key, 'Accept': 'application/json'}
    all_candles = []
    now = int(time.time())
    start = now - days_back * 86400

    t = start
    while t < now:
        chunk_end = min(t + chunk_days * 86400, now)
        params = {
            'address': mint,
            'type': '1D',
            'time_from': t,
            'time_to': chunk_end,
        }
        try:
            r = requests.get(url, headers=headers, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            items = data.get('data', {}).get('items', [])
            all_candles.extend(items)
            print(f"  Fetched {len(items)} candles from {datetime.fromtimestamp(t, tz=timezone.utc).date()} "
                  f"to {datetime.fromtimestamp(chunk_end, tz=timezone.utc).date()}")
        except Exception as e:
            print(f"  Warning: chunk failed ({e}), continuing...")
        t = chunk_end
        time.sleep(0.3)

    # Deduplicate by timestamp
    seen = set()
    unique = []
    for c in all_candles:
        ts = c.get('unixTime', c.get('time', 0))
        if ts not in seen:
            seen.add(ts)
            unique.append(c)
    unique.sort(key=lambda c: c.get('unixTime', c.get('time', 0)))
    return unique

print("Fetching SOL/USDC daily data from Birdeye...")
raw_candles = fetch_birdeye_ohlcv(SOL_MINT, BIRDEYE_KEY, days_back=365, chunk_days=90)
print(f"Total candles fetched: {len(raw_candles)}")

# Parse into arrays
timestamps = []
closes = []
highs = []
lows = []
opens = []
volumes = []

for c in raw_candles:
    ts = c.get('unixTime', c.get('time', 0))
    timestamps.append(datetime.fromtimestamp(ts, tz=timezone.utc))
    closes.append(float(c.get('c', c.get('close', 0))))
    highs.append(float(c.get('h', c.get('high', 0))))
    lows.append(float(c.get('l', c.get('low', 0))))
    opens.append(float(c.get('o', c.get('open', 0))))
    volumes.append(float(c.get('v', c.get('volume', 0))))

dates = np.array(timestamps)
close_prices = np.array(closes)
high_prices = np.array(highs)
low_prices = np.array(lows)
open_prices = np.array(opens)
vol_arr = np.array(volumes)

# Filter out zero/invalid prices
valid = close_prices > 0
dates = dates[valid]
close_prices = close_prices[valid]
high_prices = high_prices[valid]
low_prices = low_prices[valid]
open_prices = open_prices[valid]
vol_arr = vol_arr[valid]

print(f"Valid data points: {len(close_prices)}")
if len(close_prices) > 0:
    print(f"Date range: {dates[0].date()} to {dates[-1].date()}")
    print(f"Price range: ${close_prices.min():.2f} -- ${close_prices.max():.2f}")


### 1.3 Fetch Current Price from Orca Whirlpool

In [ ]:
def fetch_orca_price(rpc_url, account_pubkey):
    """Fetch current SOL/USDC price from Orca Whirlpool account."""
    payload = {
        'jsonrpc': '2.0',
        'id': 1,
        'method': 'getAccountInfo',
        'params': [account_pubkey, {'encoding': 'base64'}]
    }
    try:
        r = requests.post(rpc_url, json=payload, timeout=15)
        r.raise_for_status()
        data = r.json()
        account_data = base64.b64decode(data['result']['value']['data'][0])
        # sqrtPriceX64 is at bytes 65:81 (u128, little-endian)
        sqrt_price_x64 = int.from_bytes(account_data[65:81], 'little')
        price = (sqrt_price_x64 / (2**64))**2 * 1e3
        return price
    except Exception as e:
        print(f"  Warning: Orca fetch failed ({e}), using last Birdeye close")
        return None

current_price_orca = fetch_orca_price(HELIUS_RPC, WHIRLPOOL_ACCOUNT)
if current_price_orca is not None:
    print(f"Current SOL/USDC from Orca Whirlpool: ${current_price_orca:.4f}")
    S0 = current_price_orca
else:
    S0 = close_prices[-1] if len(close_prices) > 0 else 130.0
    print(f"Using last Birdeye close: ${S0:.4f}")

print(f"\nReference price S0 = ${S0:.2f}")


### 1.4 Realized Volatility

In [ ]:
def rolling_volatility(prices, window):
    """Compute annualized rolling volatility from log returns."""
    log_ret = np.log(prices[1:] / prices[:-1])
    vol = np.full(len(prices), np.nan)
    for i in range(window, len(log_ret) + 1):
        vol[i] = np.std(log_ret[i - window:i]) * np.sqrt(365)
    return vol

if len(close_prices) >= 90:
    vol_7d = rolling_volatility(close_prices, 7)
    vol_30d = rolling_volatility(close_prices, 30)
    vol_90d = rolling_volatility(close_prices, 90)

    log_returns = np.log(close_prices[1:] / close_prices[:-1])
    vol_full = np.std(log_returns) * np.sqrt(365)

    print("=== Realized Volatility Summary ===")
    print(f"  7-day trailing:  {np.nanmean(vol_7d):.1%} (mean), current: {vol_7d[~np.isnan(vol_7d)][-1]:.1%}")
    print(f"  30-day trailing: {np.nanmean(vol_30d):.1%} (mean), current: {vol_30d[~np.isnan(vol_30d)][-1]:.1%}")
    print(f"  90-day trailing: {np.nanmean(vol_90d):.1%} (mean), current: {vol_90d[~np.isnan(vol_90d)][-1]:.1%}")
    print(f"  Full-year:       {vol_full:.1%}")

    # Plot volatility
    fig, ax = plt.subplots(figsize=(14, 5))
    valid_idx_30 = ~np.isnan(vol_30d)
    valid_idx_90 = ~np.isnan(vol_90d)
    valid_idx_7 = ~np.isnan(vol_7d)
    ax.plot(dates[valid_idx_7], vol_7d[valid_idx_7], alpha=0.5, label='7d vol', linewidth=0.8)
    ax.plot(dates[valid_idx_30], vol_30d[valid_idx_30], label='30d vol', linewidth=1.2)
    ax.plot(dates[valid_idx_90], vol_90d[valid_idx_90], label='90d vol', linewidth=1.2)
    ax.axhline(vol_full, color='grey', ls='--', alpha=0.6, label=f'Full-year: {vol_full:.1%}')
    ax.set_title('SOL/USDC Realized Volatility (Annualized)')
    ax.set_ylabel('Volatility')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(DATA_DIR, 'volatility_rolling.png'), dpi=150)
    plt.close(fig)
    print("\nSaved: data/volatility_rolling.png")
else:
    print(f"Not enough data ({len(close_prices)} points) for volatility computation.")
    vol_full = 0.60  # fallback
    vol_7d = np.full(len(close_prices), vol_full)
    vol_30d = np.full(len(close_prices), vol_full)
    vol_90d = np.full(len(close_prices), vol_full)


### 1.5 Market Summary

In [ ]:
if len(close_prices) > 0:
    print("=" * 60)
    print("MARKET SUMMARY")
    print("=" * 60)
    print(f"  Data points:      {len(close_prices)}")
    print(f"  Date range:       {dates[0].date()} to {dates[-1].date()}")
    print(f"  Current price:    ${S0:.2f}")
    print(f"  52-week high:     ${close_prices.max():.2f}")
    print(f"  52-week low:      ${close_prices.min():.2f}")
    print(f"  Mean price:       ${close_prices.mean():.2f}")
    print(f"  Annual vol:       {vol_full:.1%}")
    total_return = (close_prices[-1] / close_prices[0]) - 1
    print(f"  1Y total return:  {total_return:.1%}")
    print("=" * 60)

    # Price plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})
    ax1.plot(dates, close_prices, 'b-', linewidth=1)
    ax1.fill_between(dates, low_prices, high_prices, alpha=0.1, color='blue')
    ax1.set_title('SOL/USDC Daily Price (1 Year)')
    ax1.set_ylabel('Price (USDC)')
    ax1.grid(True, alpha=0.3)

    ax2.bar(dates, vol_arr / 1e6, width=0.8, alpha=0.5, color='steelblue')
    ax2.set_ylabel('Volume (M USDC)')
    ax2.set_xlabel('Date')
    ax2.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(os.path.join(DATA_DIR, 'price_and_volume.png'), dpi=150)
    plt.close(fig)
    print("Saved: data/price_and_volume.png")


## Part II: Theoretical Framework

### Geometric Brownian Motion (GBM)

We model SOL/USDC price under GBM:

    S_T = S_0 * exp( (mu - sigma^2/2)*T + sigma*sqrt(T)*Z ),  Z ~ N(0,1)

For risk-neutral pricing we set mu = 0 (crypto, no carry).

### Concentrated Liquidity Value Function

For a Uniswap-v3 / Orca Whirlpool CL position with liquidity L in range [p_l, p_u]:

    V(S) = a(S)*S + b(S)

where:
    if S <= p_l:  a = L*(sqrt(p_u)-sqrt(p_l))/(sqrt(p_l)*sqrt(p_u)),  b = 0
    if S >= p_u:  a = 0,  b = L*(sqrt(p_u)-sqrt(p_l))
    otherwise:    a = L*(sqrt(p_u)-sqrt(S))/(sqrt(S)*sqrt(p_u)),  b = L*(sqrt(S)-sqrt(p_l))

### Corridor Certificate Payoff

    Payoff = min(Cap, max(0, V(S_0) - V(max(S_T, Barrier))))

This compensates for impermanent loss when price drops below the entry,
capped at Cap (natural cap = V(S_0) - V(Barrier)), with barrier at lower tick.

### Two-Part Premium Model (v2)

    P_total = alpha * FairValue * vol_indicator + beta * actual_fees

where vol_indicator = clip(sigma_7d / sigma_30d, 0.5, 2.0),
and beta is calibrated so that E[P_total] = MARKUP * FairValue.

This aligns LP and RT incentives: LP pays less upfront when vol is low,
and the settlement component ties RT compensation to actual fee generation.

### 2.1 Core Functions

In [ ]:
# ---- CL value function (vectorized) ----
def cl_value_vec(S, L, p_l, p_u):
    """Concentrated liquidity position value at price S."""
    S = np.atleast_1d(np.asarray(S, float))
    spl, spu = np.sqrt(p_l), np.sqrt(p_u)
    sp = np.sqrt(np.clip(S, 1e-10, None))
    bl = S <= p_l
    ab = S >= p_u
    a = np.where(bl, L * (spu - spl) / (spl * spu),
                 np.where(ab, 0.0, L * (spu - sp) / (sp * spu)))
    b = np.where(bl, 0.0,
                 np.where(ab, L * (spu - spl), L * (sp - spl)))
    return a * S + b


# ---- Corridor payoff (vectorized) ----
def corridor_payoff_vec(S_T, S0, B, Cap, L, p_l, p_u):
    """Corridor certificate payoff given terminal price(s) S_T."""
    S_T = np.atleast_1d(np.asarray(S_T, float))
    V0 = cl_value_vec(S0, L, p_l, p_u)
    V_eff = cl_value_vec(np.maximum(S_T, B), L, p_l, p_u)
    return np.minimum(Cap, np.maximum(0.0, np.where(S_T >= S0, 0.0, V0 - V_eff)))


# ---- Make position ----
def make_position(S0, width, L=N_LIQUIDITY):
    """Create position parameters for given width around S0."""
    p_l = S0 * (1.0 - width)
    p_u = S0 * (1.0 + width)
    V0 = float(cl_value_vec(S0, L, p_l, p_u))
    barrier = p_l  # barrier at lower tick
    V_barrier = float(cl_value_vec(barrier, L, p_l, p_u))
    natural_cap = V0 - V_barrier
    return {
        'S0': S0, 'p_l': p_l, 'p_u': p_u, 'L': L,
        'barrier': barrier, 'natural_cap': natural_cap, 'V0': V0,
        'width': width,
    }


# ---- Gauss-Hermite quadrature for fair value ----
def fv_quadrature(pos, sigma, T=T_WEEK, n_nodes=128):
    """Fair value via Gauss-Hermite quadrature (128 nodes)."""
    nodes, weights = hermgauss(n_nodes)
    # Transform: S_T = S0 * exp((- sigma^2/2)*T + sigma*sqrt(T) * z * sqrt(2))
    # hermgauss (physicist's) uses weight exp(-x^2), so z = node * sqrt(2)
    S0 = pos['S0']
    drift = -0.5 * sigma**2 * T
    diffusion = sigma * np.sqrt(T) * nodes * np.sqrt(2.0)
    S_T = S0 * np.exp(drift + diffusion)

    payoffs = corridor_payoff_vec(S_T, S0, pos['barrier'], pos['natural_cap'],
                                  pos['L'], pos['p_l'], pos['p_u'])
    # Gauss-Hermite: integral of f(x)*exp(-x^2) dx ~ sum(w_i * f(x_i))
    fv = np.sum(weights * payoffs) / np.sqrt(np.pi)
    return max(fv, 0.0)


# ---- Black-Scholes put price ----
def bs_put(S, K, sigma, T, r=0.0):
    """European put price under Black-Scholes."""
    if T <= 0 or sigma <= 0:
        return max(K - S, 0.0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-r * T) * norm.cdf(-d1)


# ---- CL delta (for perp hedge) ----
def cl_delta(S, L, p_l, p_u):
    """dV/dS of the CL position."""
    S = float(S)
    if S <= p_l:
        return L * (np.sqrt(p_u) - np.sqrt(p_l)) / (np.sqrt(p_l) * np.sqrt(p_u))
    elif S >= p_u:
        return 0.0
    else:
        sp = np.sqrt(S)
        spu = np.sqrt(p_u)
        # d(a*S + b)/dS = a + S*da/dS + db/dS
        # a = L*(spu - sp)/(sp*spu), b = L*(sp - spl)
        # Simplified: delta = L * (spu - sp) / (sp * spu) + S * d(a)/dS + d(b)/dS
        # = L * spu / (2 * sp * spu) = L / (2 * sp)  ... actually let's use numerical
        eps = S * 1e-6
        v_up = float(cl_value_vec(S + eps, L, p_l, p_u))
        v_dn = float(cl_value_vec(S - eps, L, p_l, p_u))
        return (v_up - v_dn) / (2 * eps)


# ---- Integer sqrt (Solana-style, for reference) ----
def integer_sqrt(n):
    """Integer square root (floor)."""
    if n < 0:
        raise ValueError("negative")
    if n == 0:
        return 0
    x = n
    y = (x + 1) // 2
    while y < x:
        x = y
        y = (x + n // x) // 2
    return x


# ---- Heuristic premium (on-chain style) ----
def heuristic_premium(cap, severity_ppm, sigma, T, u_ratio=0.15,
                      carry_bps=50, stress=False):
    """
    On-chain heuristic premium:
    Premium = E[Payout] + C_cap + C_adv + C_rep
    """
    width_bps = 500  # rough proxy
    p_hit = min(1.0, sigma * np.sqrt(T) / (width_bps / BPS))
    e_payout = cap * p_hit * severity_ppm / PPM / PPM * PPM  # simplify: cap * p_hit * severity/PPM
    e_payout = cap * p_hit * (severity_ppm / PPM)

    c_cap = cap * (u_ratio ** 2) / 5.0
    c_adv = (cap / 10.0) if stress else 0.0
    tenor_days = T * 365
    c_rep = cap * carry_bps * tenor_days / BPS / 100.0

    return e_payout + c_cap + c_adv + c_rep


print("Core functions defined successfully.")
print(f"  cl_value_vec, corridor_payoff_vec, make_position, fv_quadrature,")
print(f"  bs_put, cl_delta, integer_sqrt, heuristic_premium")


### 2.2 Position Setup

In [ ]:
print("=" * 70)
print("POSITION PARAMETERS")
print("=" * 70)
print(f"{'Parameter':<25} {'+-5%':>20} {'+-10%':>20}")
print("-" * 70)

positions = {}
for key, wcfg in WIDTHS.items():
    pos = make_position(S0, wcfg['width'])
    positions[key] = pos

    if key == '5pct':
        col1 = pos
    else:
        col2 = pos

rows = [
    ('Entry price S0', f"${col1['S0']:.2f}", f"${col2['S0']:.2f}"),
    ('Lower tick p_l', f"${col1['p_l']:.2f}", f"${col2['p_l']:.2f}"),
    ('Upper tick p_u', f"${col1['p_u']:.2f}", f"${col2['p_u']:.2f}"),
    ('Barrier', f"${col1['barrier']:.2f}", f"${col2['barrier']:.2f}"),
    ('Position value V0', f"${col1['V0']:.2f}", f"${col2['V0']:.2f}"),
    ('Natural cap', f"${col1['natural_cap']:.2f}", f"${col2['natural_cap']:.2f}"),
    ('Liquidity L', f"{col1['L']:,}", f"{col2['L']:,}"),
]
for label, v1, v2 in rows:
    print(f"{label:<25} {v1:>20} {v2:>20}")
print("=" * 70)


### 2.3 Real Option Market Data (Binance + Bybit)

We fetch live SOL put option prices from **both Binance and Bybit** — the only
two exchanges currently offering SOL options. We compare bid/ask execution costs
and select the cheapest available venue for each width.

This gives us the true real-world cost of the put spread alternative.

In [ ]:
# ── Real SOL option prices: Binance + Bybit ──
# We query both venues and use verified costs from direct API inspection.
# SOL options are only available on Binance (46 puts) and Bybit (131 puts).
# No other venue (Deribit, OKX, Aevo, Derive, Zeta) offers SOL options.

import requests as _req
import time as _time

def fetch_bybit_put_spread(strike_atm, strike_otm, target_dte=7):
    """Fetch put spread cost from Bybit (tighter spreads than Binance)."""
    try:
        resp = _req.get('https://api.bybit.com/v5/market/tickers',
                       params={'category': 'option', 'baseCoin': 'SOL'}, timeout=15)
        if not resp.ok: return None
        tickers = {t['symbol']: t for t in resp.json().get('result', {}).get('list', [])
                   if '-P' in t.get('symbol', '')}

        # Group by expiry, find ~7d
        from collections import defaultdict
        from datetime import datetime
        by_exp = defaultdict(dict)
        for sym, t in tickers.items():
            parts = sym.split('-')
            if len(parts) >= 4:
                try:
                    strike = float(parts[2])
                    by_exp[parts[1]][strike] = t
                except: pass

        best_cost = None; best_info = None
        for exp_str, strikes_dict in by_exp.items():
            try:
                exp_dt = datetime.strptime(exp_str, '%d%b%y')
                dte = (exp_dt.timestamp() - _time.time()) / 86400
            except: continue
            if dte < 3 or dte > 14: continue

            # Find closest strikes
            atm_strike = min(strikes_dict.keys(), key=lambda s: abs(s - strike_atm), default=None)
            otm_strike = min(strikes_dict.keys(), key=lambda s: abs(s - strike_otm), default=None)
            if atm_strike is None or otm_strike is None: continue
            if abs(atm_strike - strike_atm) > 4 or abs(otm_strike - strike_otm) > 4: continue

            atm_t = strikes_dict[atm_strike]
            otm_t = strikes_dict[otm_strike]
            atm_ask = float(atm_t.get('ask1Price', 0) or 0)
            otm_bid = float(otm_t.get('bid1Price', 0) or 0)
            atm_mark = float(atm_t.get('markPrice', 0) or 0)
            otm_mark = float(otm_t.get('markPrice', 0) or 0)
            atm_iv = float(atm_t.get('markIv', 0) or 0) * 100

            if atm_ask <= 0: continue
            cost = atm_ask - max(otm_bid, 0)

            if best_cost is None or abs(dte - target_dte) < abs(best_info['dte'] - target_dte):
                best_cost = cost
                best_info = {'dte': dte, 'exp': exp_str,
                             'atm_strike': atm_strike, 'atm_ask': atm_ask, 'atm_mark': atm_mark, 'atm_iv': atm_iv,
                             'otm_strike': otm_strike, 'otm_bid': otm_bid, 'otm_mark': otm_mark,
                             'venue': 'Bybit'}
        return best_cost, best_info
    except Exception as e:
        print(f'  Bybit error: {e}')
        return None, None

def fetch_binance_put_spread(strike_atm, strike_otm, target_dte=7):
    """Fetch put spread cost from Binance."""
    try:
        resp = _req.get('https://eapi.binance.com/eapi/v1/exchangeInfo', timeout=15)
        if not resp.ok: return None, None
        puts = [s for s in resp.json().get('optionSymbols', [])
                if 'SOL' in s.get('underlying', '') and s.get('side') == 'PUT']
        resp_t = _req.get('https://eapi.binance.com/eapi/v1/ticker', timeout=15)
        tickers = {t['symbol']: t for t in resp_t.json() if '-P' in t.get('symbol', '')} if resp_t.ok else {}
        resp_m = _req.get('https://eapi.binance.com/eapi/v1/mark', timeout=15)
        marks = {m['symbol']: m for m in resp_m.json() if 'SOL' in m.get('symbol', '')} if resp_m.ok else {}

        from collections import defaultdict
        now_ms = int(_time.time() * 1000)
        by_exp = defaultdict(list)
        for p in puts: by_exp[int(p.get('expiryDate', 0))].append(p)

        best_cost = None; best_info = None
        for exp_ts, exp_puts in by_exp.items():
            dte = (exp_ts - now_ms) / 86400000
            if dte < 3 or dte > 14: continue
            atm_sym = min(exp_puts, key=lambda p: abs(float(p['strikePrice']) - strike_atm), default=None)
            otm_sym = min(exp_puts, key=lambda p: abs(float(p['strikePrice']) - strike_otm), default=None)
            if not atm_sym or not otm_sym: continue
            atm_ask = float(tickers.get(atm_sym['symbol'], {}).get('askPrice', 0) or 0)
            otm_bid = float(tickers.get(otm_sym['symbol'], {}).get('bidPrice', 0) or 0)
            atm_mark = float(marks.get(atm_sym['symbol'], {}).get('markPrice', 0) or 0)
            if atm_ask <= 0: continue
            cost = atm_ask - max(otm_bid, 0)
            if best_cost is None or abs(dte - target_dte) < abs(best_info['dte'] - target_dte):
                best_cost = cost
                best_info = {'dte': dte, 'atm_ask': atm_ask, 'otm_bid': otm_bid,
                             'atm_mark': atm_mark, 'venue': 'Binance'}
        return best_cost, best_info
    except Exception as e:
        print(f'  Binance error: {e}')
        return None, None

# ── Fetch from both venues, pick cheapest ──
print('Fetching SOL options from Binance and Bybit...\n')

REAL_PUT_SPREAD = {}
for width_key, wcfg in WIDTHS.items():
    w = wcfg['width']
    strike_atm = round(S0)
    strike_otm = round(S0 * (1 - w))

    bb_cost, bb_info = fetch_bybit_put_spread(strike_atm, strike_otm)
    bn_cost, bn_info = fetch_binance_put_spread(strike_atm, strike_otm)

    print(f'  {wcfg["label"]}: ATM=${strike_atm}, OTM=${strike_otm}')
    if bb_cost: print(f'    Bybit:   ${bb_cost:.2f}/SOL (ask-bid execution, {bb_info["dte"]:.0f}d, IV={bb_info.get("atm_iv","?"):.0f}%)')
    else: print(f'    Bybit:   unavailable')
    if bn_cost: print(f'    Binance: ${bn_cost:.2f}/SOL (ask-bid execution, {bn_info["dte"]:.0f}d)')
    else: print(f'    Binance: unavailable')

    # Pick cheapest
    candidates = []
    if bb_cost: candidates.append(('Bybit', bb_cost, bb_info))
    if bn_cost: candidates.append(('Binance', bn_cost, bn_info))

    if candidates:
        venue, cost, info = min(candidates, key=lambda x: x[1])
        cost_pos = cost * (positions[width_key]['V0'] / S0)
        REAL_PUT_SPREAD[width_key] = {
            'cost_per_sol': cost, 'cost_position': cost_pos,
            'venue': venue, 'info': info,
        }
        # BS comparison
        sigma_current = vol_30d[~np.isnan(vol_30d)][-1] if np.any(~np.isnan(vol_30d)) else 0.65
        bs_cost = bs_put(S0, strike_atm, sigma_current, T_WEEK) - bs_put(S0, strike_otm, sigma_current, T_WEEK)
        bs_pos = bs_cost * (positions[width_key]['V0'] / S0)
        ratio = cost / max(bs_cost, 0.01)
        print(f'    >>> Best: {venue} at ${cost:.2f}/SOL = ${cost_pos:.0f}/wk (BS=${bs_pos:.0f}, ratio={ratio:.1f}x)')
    else:
        REAL_PUT_SPREAD[width_key] = None
        print(f'    >>> No options available')
    print()
# ── Dual-Source IV: Use LOWER IV from Binance and Bybit ──
print('\n--- Dual-Source IV Comparison ---')

# Extract IV from each venue's ATM puts
binance_ivs = []
bybit_ivs = []

# Bybit: markIv is a decimal (0.60 = 60%)
for width_key in ['5pct', '10pct']:
    ps = REAL_PUT_SPREAD.get(width_key)
    if ps and ps.get('info'):
        info = ps['info']
        if info.get('venue') == 'Bybit' and info.get('atm_iv', 0) > 1:
            bybit_ivs.append(info['atm_iv'] / 100.0)  # convert % to decimal
        elif info.get('venue') == 'Binance':
            # Binance markIV is already decimal (0.60 = 60%)
            if info.get('atm_mark_iv'):
                binance_ivs.append(info['atm_mark_iv'])

# Also try direct fetch for each venue's ATM IV
try:
    resp = _req.get('https://api.bybit.com/v5/market/tickers',
                   params={'category': 'option', 'baseCoin': 'SOL'}, timeout=10)
    if resp.ok:
        for t in resp.json().get('result', {}).get('list', []):
            if '-P' not in t.get('symbol', ''): continue
            parts = t['symbol'].split('-')
            if len(parts) < 4: continue
            strike = float(parts[2])
            iv = float(t.get('markIv', 0) or 0)
            if abs(strike - S0) < 3 and iv > 0.01:
                from datetime import datetime as _dtcls
                try:
                    dte = (_dtcls.strptime(parts[1], '%d%b%y').timestamp() - _time.time()) / 86400
                    if 3 < dte < 14:
                        bybit_ivs.append(iv)  # already decimal
                except: pass
except: pass

try:
    resp = _req.get('https://eapi.binance.com/eapi/v1/mark', timeout=10)
    if resp.ok:
        for m in resp.json():
            if 'SOL' not in m.get('symbol', '') or '-P' not in m.get('symbol', ''): continue
            iv_raw = float(m.get('markIV', 0) or 0)
            # Binance markIV is decimal (0.60 = 60%)
            if iv_raw > 0.05 and iv_raw < 5.0:
                # Check if ATM
                parts = m['symbol'].split('-')
                if len(parts) >= 3:
                    try:
                        strike = float(parts[2])
                        if abs(strike - S0) < 3:
                            binance_ivs.append(iv_raw)
                    except: pass
except: pass

# Compute averages per venue
bybit_iv_avg = np.mean(bybit_ivs) if bybit_ivs else None
binance_iv_avg = np.mean(binance_ivs) if binance_ivs else None
rv_30d = vol_30d[~np.isnan(vol_30d)][-1] if np.any(~np.isnan(vol_30d)) else 0.65

print(f'  Bybit ATM IV:   {bybit_iv_avg*100:.1f}%' if bybit_iv_avg else '  Bybit ATM IV:   unavailable')
print(f'  Binance ATM IV: {binance_iv_avg*100:.1f}%' if binance_iv_avg else '  Binance ATM IV: unavailable')
print(f'  RV (30d):       {rv_30d*100:.1f}%')

# Pick the LOWER IV (more competitive for LP)
candidates = []
if bybit_iv_avg and bybit_iv_avg > 0.05: candidates.append(('Bybit', bybit_iv_avg))
if binance_iv_avg and binance_iv_avg > 0.05: candidates.append(('Binance', binance_iv_avg))

if candidates:
    best_source, best_iv = min(candidates, key=lambda x: x[1])
    live_iv_rv = best_iv / max(rv_30d, 0.01)
    print(f'  >>> Using {best_source} IV = {best_iv*100:.1f}% (lower of the two)')
    print(f'  >>> IV/RV ratio = {live_iv_rv:.3f}')
else:
    best_iv = rv_30d * 1.25  # fallback estimate
    live_iv_rv = 1.25
    best_source = 'estimate'
    print(f'  >>> No live IV available, using estimate: IV/RV = 1.25')

LIVE_IV_RV = {
    'iv': best_iv, 'rv': rv_30d, 'iv_rv_ratio': live_iv_rv,
    'source': best_source,
    'bybit_iv': bybit_iv_avg, 'binance_iv': binance_iv_avg,
}

# Update today_snapshot scenario
IV_RV_SCENARIOS['today_snapshot'] = lambda s7, s30, stress, _r=live_iv_rv: _r
print(f'  IV_RV_SCENARIOS[today_snapshot] set to {live_iv_rv:.3f}')


In [ ]:
# ── IV/RV Adaptive Markup Helpers ──
def compute_adaptive_markup_for_week(sigma7d, sigma30d, stress_flag, scenario_fn, floor=DEFAULT_MARKUP_FLOOR):
    if scenario_fn is None:
        iv_rv = 1.25
    else:
        iv_rv = scenario_fn(sigma7d, sigma30d, stress_flag)
    return adaptive_markup(iv_rv, floor), iv_rv

print(f'IV/RV helpers defined. Scenarios: {list(IV_RV_SCENARIOS.keys())}')
print(f'Floors to test: {MARKUP_FLOOR_VALUES}')


## Part III: Fair Value & Heuristic Premium

### 3.1 Gauss-Hermite Fair Value

In [ ]:
sigma_current = vol_30d[~np.isnan(vol_30d)][-1] if np.any(~np.isnan(vol_30d)) else vol_full

print("=== Gauss-Hermite Fair Value (128-node) ===")
print(f"Using sigma = {sigma_current:.1%} (trailing 30d)")
print()

fv_results = {}
for key, wcfg in WIDTHS.items():
    pos = positions[key]
    fv = fv_quadrature(pos, sigma_current)
    fv_pct = fv / pos['V0'] * 100
    fv_results[key] = fv
    print(f"  {wcfg['label']:>6} width:  FV = ${fv:.4f}  ({fv_pct:.3f}% of position value)")
    print(f"           Natural cap = ${pos['natural_cap']:.4f}")
    print(f"           FV / cap    = {fv / pos['natural_cap']:.3f}")
    print()


### 3.2 Monte Carlo Validation

In [ ]:
def mc_fair_value(pos, sigma, T=T_WEEK, n_paths=200_000, antithetic=True):
    """Monte Carlo fair value with antithetic variates."""
    rng = np.random.default_rng(42)
    n_half = n_paths // 2 if antithetic else n_paths
    Z = rng.standard_normal(n_half)
    if antithetic:
        Z = np.concatenate([Z, -Z])

    S_T = pos['S0'] * np.exp((-0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = corridor_payoff_vec(S_T, pos['S0'], pos['barrier'],
                                  pos['natural_cap'], pos['L'], pos['p_l'], pos['p_u'])
    fv = np.mean(payoffs)
    se = np.std(payoffs) / np.sqrt(len(payoffs))
    return fv, se

print("=== Monte Carlo Validation (200k paths, antithetic) ===")
print(f"sigma = {sigma_current:.1%}")
print()
print(f"{'Width':>8}  {'GH FV':>10}  {'MC FV':>10}  {'MC SE':>10}  {'Diff':>8}")
print("-" * 55)

for key, wcfg in WIDTHS.items():
    pos = positions[key]
    fv_gh = fv_results[key]
    fv_mc, se_mc = mc_fair_value(pos, sigma_current)
    diff_pct = (fv_mc - fv_gh) / fv_gh * 100 if fv_gh > 0 else 0
    print(f"{wcfg['label']:>8}  ${fv_gh:>9.4f}  ${fv_mc:>9.4f}  ${se_mc:>9.6f}  {diff_pct:>+7.2f}%")

print()
print("GH and MC should agree within ~0.5%. GH is faster and used throughout.")


### 3.3 Heuristic Comparison & Dynamic Severity Calibration

In [ ]:
print("=== Heuristic vs GH Fair Value ===")
print()

for key, wcfg in WIDTHS.items():
    pos = positions[key]
    fv_gh = fv_results[key]
    h_prem = heuristic_premium(pos['natural_cap'], wcfg['severity_ppm'],
                                sigma_current, T_WEEK)
    ratio = h_prem / fv_gh if fv_gh > 0 else float('inf')
    print(f"  {wcfg['label']:>6}:  GH FV = ${fv_gh:.4f},  Heuristic = ${h_prem:.4f},  ratio = {ratio:.3f}")

    # Calibrate severity so heuristic matches GH
    # Simple bisection
    lo, hi = 100_000, 900_000
    for _ in range(50):
        mid = (lo + hi) // 2
        h_mid = heuristic_premium(pos['natural_cap'], mid, sigma_current, T_WEEK)
        if h_mid < fv_gh:
            lo = mid
        else:
            hi = mid
    calibrated_sev = (lo + hi) // 2
    h_cal = heuristic_premium(pos['natural_cap'], calibrated_sev, sigma_current, T_WEEK)
    print(f"           Calibrated severity_ppm = {calibrated_sev:,}  -> Heuristic = ${h_cal:.4f}")
    print()


### 3.4 Sensitivity: FV vs Heuristic across Sigma

In [ ]:
sigmas = np.linspace(0.20, 1.50, 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (key, wcfg) in enumerate(WIDTHS.items()):
    pos = positions[key]
    ax = axes[idx]

    fv_gh_arr = [fv_quadrature(pos, s) for s in sigmas]
    h_arr = [heuristic_premium(pos['natural_cap'], wcfg['severity_ppm'], s, T_WEEK) for s in sigmas]
    markup_arr = [fv * MARKUP for fv in fv_gh_arr]

    ax.plot(sigmas, fv_gh_arr, 'b-', linewidth=2, label='GH Fair Value')
    ax.plot(sigmas, h_arr, 'r--', linewidth=1.5, label='Heuristic')
    ax.plot(sigmas, markup_arr, 'g:', linewidth=1.5, label=f'{MARKUP}x Fair Value')
    ax.axvline(sigma_current, color='grey', ls=':', alpha=0.6, label=f'Current vol ({sigma_current:.0%})')
    ax.set_title(f'{wcfg["label"]} Width')
    ax.set_xlabel('Annualized Volatility')
    ax.set_ylabel('Premium (USDC)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Fair Value vs Heuristic Premium across Volatility', fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(DATA_DIR, 'fv_vs_heuristic.png'), dpi=150)
plt.close(fig)
print("Saved: data/fv_vs_heuristic.png")


## Part IV: Two-Part Premium Model

### The Model

    P_total = alpha * FairValue * vol_indicator + beta * fees_actual

Where:
- alpha = 0.40 (40% paid upfront by LP)
- vol_indicator = clip(sigma_7d / sigma_30d, 0.5, 2.0) -- recent vol vs trend
- beta = max(0, (MARKUP - alpha) * FV / expected_weekly_fees)
- fees_actual = realized trading fees during the week

This means:
- In low-vol weeks: LP pays less upfront, RT gets compensated via actual fee share
- In high-vol weeks: LP pays more upfront, which is fair since risk is higher
- E[P_total] = 1.10 * FairValue (10% markup over fair value)

### 4.1 Beta Calibration

In [ ]:
print("=== Two-Part Premium: Beta Calibration ===")
print()

# Expected weekly fees for each width
for key, wcfg in WIDTHS.items():
    pos = positions[key]
    fv = fv_results[key]
    weekly_fee = wcfg['daily_fee'] * 7 * pos['V0']
    beta = max(0, (MARKUP - ALPHA) * fv / max(weekly_fee, 1e-10))
    wcfg['_beta'] = beta
    wcfg['_weekly_fee_expected'] = weekly_fee
    wcfg['_fv'] = fv

    print(f"  {wcfg['label']} width:")
    print(f"    Fair Value:           ${fv:.4f}")
    print(f"    Expected weekly fee:  ${weekly_fee:.4f}")
    print(f"    beta:                 {beta:.6f}")
    print(f"    Upfront (alpha*FV):   ${ALPHA * fv:.4f}")
    print(f"    E[settlement]:        ${beta * weekly_fee:.4f}")
    print(f"    E[total]:             ${ALPHA * fv + beta * weekly_fee:.4f}")
    print(f"    Target (1.10*FV):     ${MARKUP * fv:.4f}")
    print()


### 4.2 Premium Decomposition: Good vs Bad Weeks

In [ ]:
print("=== Premium Decomposition Examples ===")
print()

for key, wcfg in WIDTHS.items():
    pos = positions[key]
    fv = wcfg['_fv']
    beta = wcfg['_beta']
    wf_exp = wcfg['_weekly_fee_expected']

    print(f"--- {wcfg['label']} Width ---")
    print(f"{'Scenario':<25} {'Vol Ind':>8} {'Upfront':>10} {'Settl.':>10} {'Total':>10} {'vs FV':>8}")
    print("-" * 75)

    scenarios = [
        ('Low vol, good fees', 0.6, wf_exp * 1.3),
        ('Normal vol, normal fees', 1.0, wf_exp * 1.0),
        ('High vol, low fees', 1.8, wf_exp * 0.5),
        ('Stress (high vol, no fees)', 2.0, wf_exp * 0.1),
    ]
    for name, vi, fees in scenarios:
        upfront = ALPHA * fv * vi
        settl = beta * fees
        total = upfront + settl
        vs_fv = total / fv if fv > 0 else 0
        print(f"{name:<25} {vi:>8.1f} ${upfront:>9.4f} ${settl:>9.4f} ${total:>9.4f} {vs_fv:>7.2f}x")
    print()


### 4.3 Premium Components Over Simulated Weeks

In [ ]:
# Simulate premium decomposition over the real data
if len(close_prices) >= 37:  # need at least 30+7 days
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for idx, (key, wcfg) in enumerate(WIDTHS.items()):
        ax = axes[idx]
        pos = positions[key]
        fv = wcfg['_fv']
        beta = wcfg['_beta']

        weeks_upfront = []
        weeks_settl = []
        weeks_dates = []

        # Walk through data in weekly steps
        for i in range(37, len(close_prices), 7):
            # vol indicator
            v7 = vol_7d[i] if not np.isnan(vol_7d[i]) else vol_full
            v30 = vol_30d[i] if not np.isnan(vol_30d[i]) else vol_full
            vi = np.clip(v7 / max(v30, 1e-6), 0.5, 2.0)

            # Recompute FV at trailing vol
            local_pos = make_position(close_prices[i - 7], wcfg['width'])
            local_fv = fv_quadrature(local_pos, v30)

            local_beta = max(0, (MARKUP - ALPHA) * local_fv /
                             max(wcfg['daily_fee'] * 7 * local_pos['V0'], 1e-10))

            upfront = ALPHA * local_fv * vi
            # Simulate fees: in-range days * daily_fee * position_value
            week_prices = close_prices[max(0, i - 7):i]
            in_range_frac = np.mean((week_prices >= local_pos['p_l']) &
                                     (week_prices <= local_pos['p_u']))
            actual_fees = in_range_frac * wcfg['daily_fee'] * 7 * local_pos['V0']
            settl = local_beta * actual_fees

            weeks_upfront.append(upfront)
            weeks_settl.append(settl)
            weeks_dates.append(dates[i])

        x = np.arange(len(weeks_dates))
        ax.bar(x, weeks_upfront, label='Upfront (alpha*FV*vi)', alpha=0.7, color='steelblue')
        ax.bar(x, weeks_settl, bottom=weeks_upfront, label='Settlement (beta*fees)',
               alpha=0.7, color='coral')
        ax.set_title(f'{wcfg["label"]} Width: Premium Decomposition')
        ax.set_xlabel('Week Index')
        ax.set_ylabel('Premium (USDC)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, axis='y')

    fig.tight_layout()
    fig.savefig(os.path.join(DATA_DIR, 'premium_decomposition.png'), dpi=150)
    plt.close(fig)
    print("Saved: data/premium_decomposition.png")
else:
    print("Not enough data for weekly decomposition plot.")


## Part V: Strategy Evaluation (Real Data Backtest)

### 5.0 Strategy Definitions

We evaluate 9 strategies on a weekly rolling basis using real price data:

1. **Bond** -- 12% APY risk-free benchmark
2. **Plain LP** -- unhedged concentrated liquidity
3. **Hedged LP (fixed premium 1.10x)** -- corridor cert at fixed 110% of FV
4. **Hedged LP (two-part premium)** -- alpha=0.4 upfront + beta*fees settlement
5. **Hedged LP + jitoSOL (two-part)** -- staking yield on SOL portion
6. **RT v1 (Pure Insurer, two-part)** -- collects premiums, pays corridor claims
7. **RT v2 (Productive Capital, two-part)** -- lending yield + premiums - claims
8. **LP + Put Spread** -- Black-Scholes ATM/barrier put spread hedge
9. **LP + Perp Delta Hedge** -- short perp at CL delta, with funding

### 5.1 Backtest Engine

In [ ]:
def run_backtest(close_prices, dates, vol_7d, vol_30d, vol_full, widths_config,
                 positions_dict, width_key='5pct',
                 markup_mode='fixed', iv_rv_scenario_fn=None, markup_floor=DEFAULT_MARKUP_FLOOR):
    """Run weekly rolling backtest for all 10 strategies.

    All strategy returns are expressed as FRACTION OF V0 (position value)
    so that returns are comparable across widths and with bond.
    """
    wcfg = widths_config[width_key]
    fs_frac = wcfg['fee_share_bps'] / BPS
    results = {name: [] for name in [
        'bond', 'plain_lp', 'hedged_fixed', 'hedged_twopart',
        'hedged_twopart_jito', 'rt_v1', 'rt_v2', 'lp_put_spread', 'lp_put_real', 'lp_perp',
        'corridor_perp_tail'
    ]}
    week_dates = []
    week_prices_start = []
    week_prices_end = []
    week_iv_rv_ratios = []

    min_start = 37  # need 30d vol history

    for i in range(min_start, len(close_prices) - 7, 7):
        S_start = close_prices[i]
        S_end = close_prices[i + 7]
        if S_start <= 0 or S_end <= 0:
            continue

        week_dates.append(dates[i])
        week_prices_start.append(S_start)
        week_prices_end.append(S_end)

        # Recenter position each week
        pos = make_position(S_start, wcfg['width'])
        V0 = pos['V0']

        # Trailing volatility
        s30 = vol_30d[i] if i < len(vol_30d) and not np.isnan(vol_30d[i]) else vol_full
        s7 = vol_7d[i] if i < len(vol_7d) and not np.isnan(vol_7d[i]) else vol_full
        sigma = max(s30, 0.10)
        vol_ind = np.clip(s7 / max(s30, 1e-6), 0.5, 2.0)

        # Fair value
        fv = fv_quadrature(pos, sigma)

        # In-range fraction and fees
        week_slice = close_prices[i:i + 7]
        in_range = np.mean((week_slice >= pos['p_l']) & (week_slice <= pos['p_u']))
        weekly_fees = in_range * wcfg['daily_fee'] * 7 * V0

        # IL: change in position value
        V_end = float(cl_value_vec(S_end, pos['L'], pos['p_l'], pos['p_u']))
        IL = V_end - V0  # negative when IL occurs

        # Corridor payoff
        corridor_pay = float(corridor_payoff_vec(
            S_end, S_start, pos['barrier'], pos['natural_cap'],
            pos['L'], pos['p_l'], pos['p_u']))

        # ── Effective markup ──
        if markup_mode == 'adaptive' and iv_rv_scenario_fn is not None:
            stress_proxy = (s7 / max(s30, 1e-6) > 1.5)
            eff_markup, _iv_rv = compute_adaptive_markup_for_week(s7, s30, stress_proxy, iv_rv_scenario_fn, markup_floor)
            week_iv_rv_ratios.append(_iv_rv)
        else:
            eff_markup = MARKUP
            week_iv_rv_ratios.append(float('nan'))

        # Premiums
        # Fixed premium
        prem_fixed = eff_markup * fv

        # Two-part premium
        wf_exp = wcfg['daily_fee'] * 7 * V0
        beta = max(0, (eff_markup - ALPHA) * fv / max(wf_exp, 1e-10))
        prem_upfront = ALPHA * fv * vol_ind
        prem_settl = beta * weekly_fees
        prem_twopart = prem_upfront + prem_settl

        # Protocol fee
        proto_fee_fixed = prem_fixed * PROTOCOL_FEE_BPS / BPS
        proto_fee_tp = prem_twopart * PROTOCOL_FEE_BPS / BPS

        # Effective premium for corridor (LP pays premium at buy + settle tx costs)
        effective_premium = prem_twopart

        # Transaction cost: open + close (or buy + settle), ~$0.17 at $84 SOL
        tx_cost = TX_COST_SOL * S_start * 2

        # --- Strategy returns (FRACTION OF V0) ---

        # 1. Bond (compound weekly return, already a fraction)
        bond_ret = (1 + BOND_APY) ** (1 / 52) - 1
        results['bond'].append(bond_ret)

        # 2. Plain LP (unhedged)
        plain_lp_ret = (IL + weekly_fees) / V0
        results['plain_lp'].append(plain_lp_ret)

        # 3. Hedged LP -- fixed premium (buy + settle tx costs)
        eff_prem_fixed = max(0, prem_fixed - weekly_fees * fs_frac)
        hedged_fixed_ret = (IL + weekly_fees + corridor_pay - eff_prem_fixed - tx_cost) / V0
        results['hedged_fixed'].append(hedged_fixed_ret)

        # 4. Hedged LP -- two-part (buy + settle tx costs)
        hedged_tp_ret = (IL + weekly_fees + corridor_pay - prem_twopart - tx_cost) / V0
        results['hedged_twopart'].append(hedged_tp_ret)

        # 5. Hedged LP + jitoSOL -- two-part (buy + settle tx costs)
        jito_yield = V0 * SOL_FRACTION * JITO_APY * T_WEEK
        hedged_jito_ret = (IL + weekly_fees + corridor_pay - prem_twopart + jito_yield - tx_cost) / V0
        results['hedged_twopart_jito'].append(hedged_jito_ret)

        # 6. RT v1 (Pure Insurer) -- two-part
        # RT receives premium (net of protocol fee), pays corridor claims
        rt_v1_ret = ((prem_twopart - proto_fee_tp) - corridor_pay) / V0
        results['rt_v1'].append(rt_v1_ret)

        # 7. RT v2 (Productive Capital) -- two-part
        # RT also earns lending yield on deposited USDC
        # Assume RT capital = natural_cap (collateral for max payout)
        rt_capital = pos['natural_cap']
        lending_yield = rt_capital * BOND_APY * T_WEEK
        rt_v2_ret = ((prem_twopart - proto_fee_tp) - corridor_pay + lending_yield) / V0
        results['rt_v2'].append(rt_v2_ret)

        # 8. LP + Put Spread (with IV premium, bid-ask spread, and tx costs)
        sol_units = V0 * SOL_FRACTION / S_start
        sigma_iv = sigma + IV_PREMIUM
        put_atm = bs_put(S_start, S_start, sigma_iv, T_WEEK)
        put_barrier = bs_put(S_start, pos['barrier'], sigma_iv, T_WEEK)
        put_spread_cost = (put_atm - put_barrier) * sol_units
        put_spread_cost *= (1 + OPTION_SPREAD_PCT)
        put_spread_cost += tx_cost
        put_atm_pay = max(S_start - S_end, 0)
        put_bar_pay = max(pos['barrier'] - S_end, 0)
        put_spread_pay = (put_atm_pay - put_bar_pay) * sol_units
        lp_put_ret = (IL + weekly_fees + put_spread_pay - put_spread_cost) / V0
        results['lp_put_spread'].append(lp_put_ret)

        # ── Strategy: LP + Put Spread (Real Binance prices) ──
        real_ps_data = REAL_PUT_SPREAD.get(width_key)
        if real_ps_data and real_ps_data.get('cost_per_sol'):
            real_ps_cost = real_ps_data['cost_per_sol'] * (pos['V0'] / S_start)
            dte_ratio = 7.0 / max(real_ps_data['info']['dte'], 1)
            real_ps_cost *= min(dte_ratio, 2.0)
            real_ps_pay = (pos['V0'] / S_start) * max(0, max(0, S_start - S_end) - max(0, pos['barrier'] - S_end))
            lp_put_real_ret = (IL + weekly_fees + real_ps_pay - real_ps_cost - tx_cost) / V0
        else:
            real_ps_cost = put_spread_cost * 1.5
            lp_put_real_ret = (IL + weekly_fees + put_spread_pay - real_ps_cost - tx_cost) / V0
        results['lp_put_real'].append(lp_put_real_ret)

        # 9. LP + Perp Delta Hedge (with funding, fees, price impact, tx costs)
        delta = cl_delta(S_start, pos['L'], pos['p_l'], pos['p_u'])
        short_size_sol = delta
        perp_pnl = short_size_sol * (S_start - S_end)
        perp_funding = short_size_sol * S_start * PERP_FUNDING_APY * T_WEEK
        perp_fees = short_size_sol * S_start * PERP_FEE_BPS / BPS * 2
        perp_impact = short_size_sol * S_start * PERP_PRICE_IMPACT_BPS / BPS * 2
        lp_perp_ret = (IL + weekly_fees + perp_pnl - perp_funding - perp_fees - perp_impact - tx_cost) / V0
        results['lp_perp'].append(lp_perp_ret)

        # 10. Corridor + Perp Tail: corridor covers in-range IL, perp covers below-barrier
        perp_tail_pnl = 0.0
        perp_tail_cost = 0.0
        perp_open = False
        perp_sol_amount = 0.0
        perp_entry_price = 0.0

        for day_offset in range(7):
            day_idx = i + day_offset
            if day_idx >= len(close_prices):
                break
            S_day = close_prices[day_idx]
            S_next = close_prices[min(day_idx + 1, len(close_prices) - 1)]

            if S_day < pos['p_l'] and not perp_open:
                sol_amount = pos['L'] * (1/np.sqrt(pos['p_l']) - 1/np.sqrt(pos['p_u']))
                perp_sol_amount = sol_amount
                perp_entry_price = S_day
                perp_open = True

            if perp_open:
                perp_tail_pnl += -perp_sol_amount * (S_next - S_day)
                perp_tail_cost += perp_sol_amount * S_day * PERP_FUNDING_APY / 365

                if S_next >= pos['p_l']:
                    perp_tail_cost += perp_sol_amount * (perp_entry_price + S_next) * PERP_FEE_BPS / 10000
                    perp_tail_cost += perp_sol_amount * (perp_entry_price + S_next) * PERP_PRICE_IMPACT_BPS / 10000
                    perp_open = False

        if perp_open:
            perp_tail_cost += perp_sol_amount * (perp_entry_price + S_end) * PERP_FEE_BPS / 10000
            perp_tail_cost += perp_sol_amount * (perp_entry_price + S_end) * PERP_PRICE_IMPACT_BPS / 10000

        corridor_perp_tail_ret = (IL + weekly_fees + corridor_pay - effective_premium + perp_tail_pnl - perp_tail_cost - tx_cost) / V0
        results['corridor_perp_tail'].append(corridor_perp_tail_ret)

    return results, week_dates, week_prices_start, week_prices_end, week_iv_rv_ratios

print("Backtest engine defined (returns = fraction of V0). Running for both widths...")


### 5.2 Run Backtest

In [ ]:
# ── Run Backtest: Fixed + Adaptive modes ──
bt_results = {}
bt_meta = {}
bt_adaptive = {}

# --- Fixed mode (backward-compatible) ---
print("Running FIXED markup backtest...")
for width_key in ['5pct', '10pct']:
    res, wdates, wps, wpe, wivr = run_backtest(
        close_prices, dates, vol_7d, vol_30d, vol_full,
        WIDTHS, positions, width_key=width_key,
        markup_mode='fixed'
    )
    bt_results[width_key] = res
    bt_meta[width_key] = {
        'dates': wdates,
        'prices_start': wps,
        'prices_end': wpe,
        'n_weeks': len(wdates),
    }
    print(f"  {WIDTHS[width_key]['label']}: {len(wdates)} weeks (fixed MARKUP={MARKUP})")

# --- Adaptive mode: each scenario x default floor ---
print("\nRunning ADAPTIVE markup backtest...")
for width_key in ['5pct', '10pct']:
    bt_adaptive[width_key] = {}
    for sc_name, sc_fn in IV_RV_SCENARIOS.items():
        res, wdates, wps, wpe, wivr = run_backtest(
            close_prices, dates, vol_7d, vol_30d, vol_full,
            WIDTHS, positions, width_key=width_key,
            markup_mode='adaptive', iv_rv_scenario_fn=sc_fn,
            markup_floor=DEFAULT_MARKUP_FLOOR
        )
        bt_adaptive[width_key][sc_name] = {
            'results': res,
            'iv_rv_ratios': wivr,
            'dates': wdates,
        }
        med_ivrv = np.nanmedian(wivr)
        print(f"  {WIDTHS[width_key]['label']} / {sc_name}: {len(wdates)} wks, median IV/RV={med_ivrv:.2f}, floor={DEFAULT_MARKUP_FLOOR}")

print("\nBacktest complete (fixed + adaptive).")


In [ ]:
# ── Adaptive vs Fixed Comparison Table ──
print("=" * 130)
print("ADAPTIVE MARKUP vs FIXED MARKUP COMPARISON")
print("=" * 130)

for width_key in ['5pct', '10pct']:
    label = WIDTHS[width_key]['label']
    fixed_res = bt_results[width_key]

    print(f"\n{'─'*130}")
    print(f"  Width: {label}    |  Fixed MARKUP={MARKUP}")
    print(f"{'─'*130}")

    header = f"{'Mode':<35} {'LP Med/wk':>11} {'RT Med/wk':>11} {'LP Sharpe':>10} {'RT Sharpe':>10} {'LP P(+)':>8} {'Corr>PS':>8}"
    print(header)
    print("-" * 130)

    # Fixed baseline
    lp_arr = np.array(fixed_res['hedged_twopart'])
    rt_arr = np.array(fixed_res['rt_v2'])
    ps_arr = np.array(fixed_res['lp_put_spread'])
    corr_wins = np.mean(lp_arr > ps_arr) * 100 if len(lp_arr) > 0 else 0
    lp_sh = np.mean(lp_arr) / np.std(lp_arr) if np.std(lp_arr) > 0 else 0
    rt_sh = np.mean(rt_arr) / np.std(rt_arr) if np.std(rt_arr) > 0 else 0
    print(f"{'Fixed (MARKUP=' + str(MARKUP) + ')':<35} {np.median(lp_arr)*100:>+10.3f}% {np.median(rt_arr)*100:>+10.3f}% "
          f"{lp_sh:>9.3f} {rt_sh:>9.3f} {np.mean(lp_arr > 0):>7.1%} {corr_wins:>7.1f}%")

    # Adaptive scenarios
    for sc_name, sc_data in bt_adaptive[width_key].items():
        a_res = sc_data['results']
        a_lp = np.array(a_res['hedged_twopart'])
        a_rt = np.array(a_res['rt_v2'])
        a_ps = np.array(a_res['lp_put_spread'])
        a_corr = np.mean(a_lp > a_ps) * 100 if len(a_lp) > 0 else 0
        a_lp_sh = np.mean(a_lp) / np.std(a_lp) if np.std(a_lp) > 0 else 0
        a_rt_sh = np.mean(a_rt) / np.std(a_rt) if np.std(a_rt) > 0 else 0
        med_iv = np.nanmedian(sc_data['iv_rv_ratios'])
        row_label = f"Adaptive/{sc_name} (floor={DEFAULT_MARKUP_FLOOR})"
        print(f"{row_label:<35} {np.median(a_lp)*100:>+10.3f}% {np.median(a_rt)*100:>+10.3f}% "
              f"{a_lp_sh:>9.3f} {a_rt_sh:>9.3f} {np.mean(a_lp > 0):>7.1%} {a_corr:>7.1f}%")

print("\n" + "=" * 130)
print("Note: Corr>PS = % of weeks corridor hedge outperforms put spread hedge.")


### 5.3 Results Summary

In [ ]:
STRATEGY_LABELS = {
    'bond': '1. Bond (12% APY)',
    'plain_lp': '2. Plain LP (unhedged)',
    'hedged_fixed': '3. Hedged LP (fixed 1.10x)',
    'hedged_twopart': '4. Hedged LP (two-part)',
    'hedged_twopart_jito': '5. Hedged LP + jitoSOL',
    'rt_v1': '6. RT v1 (Pure Insurer)',
    'rt_v2': '7. RT v2 (Productive Capital)',
    'lp_put_spread': '8. LP + Put Spread (BS+IV)',
    'lp_put_real': '11. LP + Put Spread (Best Real Mkt)',
    'lp_perp': '9. LP + Perp Delta Hedge',
    'corridor_perp_tail': '10. Corridor + Perp Tail',
}

def compute_stats(returns_list):
    """Compute summary statistics for a list of weekly returns (fractions of V0)."""
    arr = np.array(returns_list)
    n = len(arr)
    if n == 0:
        return {}
    median_w = np.median(arr)
    mean_w = np.mean(arr)
    std_w = np.std(arr, ddof=1) if n > 1 else 0
    sharpe = mean_w / std_w if std_w > 0 else 0
    # Multiplicative cumulative wealth for drawdown
    cum_wealth = np.cumprod(1 + arr)
    running_max = np.maximum.accumulate(cum_wealth)
    drawdowns = (cum_wealth - running_max) / running_max  # fractional drawdown
    max_dd = np.min(drawdowns)  # most negative
    pct5 = np.percentile(arr, 5)
    p_pos = np.mean(arr > 0)
    # Proper compound annualization
    ann_median = (1 + median_w) ** 52 - 1
    return {
        'median_w': median_w,
        'mean_w': mean_w,
        'std_w': std_w,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'pct5': pct5,
        'p_pos': p_pos,
        'ann_median': ann_median,
        'n': n,
    }

for width_key in ['5pct', '10pct']:
    res = bt_results[width_key]
    meta = bt_meta[width_key]
    label = WIDTHS[width_key]['label']

    print()
    print("=" * 115)
    print(f"STRATEGY RESULTS -- {label} WIDTH ({meta['n_weeks']} weeks)  [returns = % of V0]")
    print("=" * 115)
    header = (f"{'Strategy':<32} {'Median/wk':>10} {'Mean/wk':>10} {'Sharpe':>8} "
              f"{'MaxDD':>10} {'5th%':>10} {'P(+)':>7} {'Ann Med':>10}")
    print(header)
    print("-" * 115)

    for skey, slabel in STRATEGY_LABELS.items():
        st = compute_stats(res[skey])
        if not st:
            continue
        print(f"{slabel:<32} {st['median_w']*100:>+9.3f}% {st['mean_w']*100:>+9.3f}% "
              f"{st['sharpe']:>7.3f} {st['max_dd']*100:>+9.2f}% {st['pct5']*100:>+9.3f}% "
              f"{st['p_pos']:>6.1%} {st['ann_median']*100:>+9.2f}%")
    print("=" * 115)


### 5.4 Head-to-Head Comparison

In [ ]:
print("=== Head-to-Head: Two-Part Hedged LP vs Alternatives ===")
print("(Count of weeks where two-part premium wins)")
print()

for width_key in ['5pct', '10pct']:
    res = bt_results[width_key]
    label = WIDTHS[width_key]['label']
    tp = np.array(res['hedged_twopart'])
    n = len(tp)

    print(f"--- {label} Width ({n} weeks) ---")
    print(f"{'Comparator':<32} {'TP wins':>8} {'Ties':>6} {'TP loses':>9} {'Win%':>7}")
    print("-" * 65)

    for skey, slabel in STRATEGY_LABELS.items():
        if skey == 'hedged_twopart':
            continue
        other = np.array(res[skey])
        wins = int(np.sum(tp > other))
        ties = int(np.sum(np.isclose(tp, other, atol=1e-8)))
        losses = n - wins - ties
        win_pct = wins / n if n > 0 else 0
        print(f"vs {slabel:<29} {wins:>8} {ties:>6} {losses:>9} {win_pct:>6.1%}")
    print()


### 5.5 Strategy Comparison Plots

In [ ]:
for width_key in ['5pct', '10pct']:
    res = bt_results[width_key]
    meta = bt_meta[width_key]
    label = WIDTHS[width_key]['label']
    wdates = meta['dates']

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Top-left: Cumulative wealth for main strategies
    ax = axes[0, 0]
    for skey, color, ls in [
        ('bond', 'grey', '--'),
        ('plain_lp', 'blue', '-'),
        ('hedged_twopart', 'green', '-'),
        ('hedged_twopart_jito', 'darkgreen', '-.'),
        ('lp_perp', 'orange', '-'),
        ('lp_put_spread', 'purple', '-'),
        ('corridor_perp_tail', 'brown', '-.'),
    ]:
        cum = np.cumprod(1 + np.array(res[skey]))
        ax.plot(range(len(cum)), cum, color=color, ls=ls, linewidth=1.2,
                label=STRATEGY_LABELS[skey])
    ax.axhline(1.0, color='black', ls=':', linewidth=0.5)
    ax.set_title(f'{label}: LP Strategy Cumulative Wealth')
    ax.set_xlabel('Week')
    ax.set_ylabel('Wealth (1 = initial)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # Top-right: RT cumulative wealth
    ax = axes[0, 1]
    for skey, color, ls in [
        ('bond', 'grey', '--'),
        ('rt_v1', 'red', '-'),
        ('rt_v2', 'darkred', '-'),
    ]:
        cum = np.cumprod(1 + np.array(res[skey]))
        ax.plot(range(len(cum)), cum, color=color, ls=ls, linewidth=1.2,
                label=STRATEGY_LABELS[skey])
    ax.axhline(1.0, color='black', ls=':', linewidth=0.5)
    ax.set_title(f'{label}: RT Strategy Cumulative Wealth')
    ax.set_xlabel('Week')
    ax.set_ylabel('Wealth (1 = initial)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Bottom-left: Weekly returns distribution (%)
    ax = axes[1, 0]
    strategies_to_hist = ['plain_lp', 'hedged_twopart', 'lp_perp']
    colors = ['blue', 'green', 'orange']
    for skey, color in zip(strategies_to_hist, colors):
        ax.hist(np.array(res[skey]) * 100, bins=30, alpha=0.4, color=color,
                label=STRATEGY_LABELS[skey], edgecolor=color, linewidth=0.5)
    ax.axvline(0, color='black', ls='-', linewidth=0.5)
    ax.set_title(f'{label}: Weekly Return Distribution')
    ax.set_xlabel('Weekly Return (% of V0)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # Bottom-right: Fixed vs Two-Part premium comparison
    ax = axes[1, 1]
    fixed_arr = np.array(res['hedged_fixed'])
    tp_arr = np.array(res['hedged_twopart'])
    diff = (tp_arr - fixed_arr) * 100  # percentage points
    colors_bar = ['green' if d > 0 else 'red' for d in diff]
    ax.bar(range(len(diff)), diff, color=colors_bar, alpha=0.6, width=1.0)
    ax.axhline(0, color='black', ls='-', linewidth=0.5)
    ax.set_title(f'{label}: Two-Part minus Fixed (weekly %)')
    ax.set_xlabel('Week')
    ax.set_ylabel('Difference (% of V0)')
    ax.grid(True, alpha=0.3)

    fig.suptitle(f'Backtest Results -- {label}', fontsize=13)
    fig.tight_layout()
    fig.savefig(os.path.join(DATA_DIR, f'backtest_{width_key}.png'), dpi=150)
    plt.close(fig)
    print(f"Saved: data/backtest_{width_key}.png")


## Part VI: LP/RT Balance Analysis

### 6.1 Viability Check

In [ ]:
print("=== LP/RT Viability Check ===")
print("Both LP and RT must have positive expected returns for the protocol to work.")
print()

for width_key in ['5pct', '10pct']:
    res = bt_results[width_key]
    label = WIDTHS[width_key]['label']
    meta = bt_meta[width_key]

    lp_mean = np.mean(res['hedged_twopart'])
    rt_mean = np.mean(res['rt_v2'])
    lp_med = np.median(res['hedged_twopart'])
    rt_med = np.median(res['rt_v2'])

    print(f"  {label} Width ({meta['n_weeks']} weeks):")
    print(f"    LP (hedged, two-part):  mean = {lp_mean*100:+.3f}%/wk,  median = {lp_med*100:+.3f}%/wk")
    print(f"    RT v2 (productive):     mean = {rt_mean*100:+.3f}%/wk,  median = {rt_med*100:+.3f}%/wk")
    viable = lp_mean > 0 and rt_mean > 0
    print(f"    Viable: {'YES' if viable else 'NO'}")
    if viable:
        print(f"    LP annualized median: {((1+lp_med)**52 - 1)*100:+.2f}%")
        print(f"    RT annualized median: {((1+rt_med)**52 - 1)*100:+.2f}%")
    print()


### 6.2 RT Return Decomposition

In [ ]:
print("=== RT Return Decomposition ===")
print()

for width_key in ['5pct', '10pct']:
    res = bt_results[width_key]
    label = WIDTHS[width_key]['label']

    # Re-run to extract components
    wcfg = WIDTHS[width_key]
    prem_collected = []
    claims_paid = []
    lending_earned = []

    min_start = 37
    for i in range(min_start, len(close_prices) - 7, 7):
        S_start = close_prices[i]
        S_end = close_prices[i + 7]
        if S_start <= 0 or S_end <= 0:
            continue

        pos = make_position(S_start, wcfg['width'])
        s30 = vol_30d[i] if i < len(vol_30d) and not np.isnan(vol_30d[i]) else vol_full
        s7 = vol_7d[i] if i < len(vol_7d) and not np.isnan(vol_7d[i]) else vol_full
        sigma = max(s30, 0.10)
        vi = np.clip(s7 / max(s30, 1e-6), 0.5, 2.0)

        fv = fv_quadrature(pos, sigma)
        wf_exp = wcfg['daily_fee'] * 7 * pos['V0']
        beta = max(0, (MARKUP - ALPHA) * fv / max(wf_exp, 1e-10))

        week_slice = close_prices[i:i + 7]
        in_range = np.mean((week_slice >= pos['p_l']) & (week_slice <= pos['p_u']))
        actual_fees = in_range * wcfg['daily_fee'] * 7 * pos['V0']

        prem = ALPHA * fv * vi + beta * actual_fees
        proto_fee = prem * PROTOCOL_FEE_BPS / BPS
        net_prem = prem - proto_fee

        corridor = float(corridor_payoff_vec(
            S_end, S_start, pos['barrier'], pos['natural_cap'],
            pos['L'], pos['p_l'], pos['p_u']))

        lending = pos['natural_cap'] * BOND_APY * T_WEEK

        prem_collected.append(net_prem)
        claims_paid.append(corridor)
        lending_earned.append(lending)

    total_prem = np.sum(prem_collected)
    total_claims = np.sum(claims_paid)
    total_lending = np.sum(lending_earned)
    n = len(prem_collected)

    print(f"--- {label} Width ({n} weeks) ---")
    print(f"  Total premiums collected:   ${total_prem:.2f}")
    print(f"  Total claims paid:          ${total_claims:.2f}")
    print(f"  Total lending yield:        ${total_lending:.2f}")
    print(f"  Net RT profit:              ${total_prem - total_claims + total_lending:.2f}")
    print(f"  Loss ratio (claims/prems):  {total_claims / total_prem:.2%}" if total_prem > 0 else "  N/A")
    print(f"  Weeks with claims > 0:      {np.sum(np.array(claims_paid) > 0)}/{n}")
    print()


### 6.3 Optimal Alpha Sensitivity Sweep

In [ ]:
alphas = np.linspace(0.0, 1.0, 21)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (width_key, wcfg) in enumerate(WIDTHS.items()):
    ax = axes[idx]
    label = wcfg['label']

    lp_means = []
    rt_means = []
    lp_sharpes = []

    for a in alphas:
        lp_rets = []
        rt_rets = []

        min_start = 37
        for i in range(min_start, len(close_prices) - 7, 7):
            S_start = close_prices[i]
            S_end = close_prices[i + 7]
            if S_start <= 0 or S_end <= 0:
                continue

            pos = make_position(S_start, wcfg['width'])
            V0 = pos['V0']
            s30 = vol_30d[i] if i < len(vol_30d) and not np.isnan(vol_30d[i]) else vol_full
            s7 = vol_7d[i] if i < len(vol_7d) and not np.isnan(vol_7d[i]) else vol_full
            sigma = max(s30, 0.10)
            vi = np.clip(s7 / max(s30, 1e-6), 0.5, 2.0)

            fv = fv_quadrature(pos, sigma)
            wf_exp = wcfg['daily_fee'] * 7 * V0
            beta = max(0, (MARKUP - a) * fv / max(wf_exp, 1e-10))

            week_slice = close_prices[i:i + 7]
            in_range_frac = np.mean((week_slice >= pos['p_l']) & (week_slice <= pos['p_u']))
            actual_fees = in_range_frac * wcfg['daily_fee'] * 7 * V0

            prem = a * fv * vi + beta * actual_fees
            proto_fee = prem * PROTOCOL_FEE_BPS / BPS

            V_end = float(cl_value_vec(S_end, pos['L'], pos['p_l'], pos['p_u']))
            IL = V_end - V0
            corridor = float(corridor_payoff_vec(
                S_end, S_start, pos['barrier'], pos['natural_cap'],
                pos['L'], pos['p_l'], pos['p_u']))

            lp_ret = (IL + actual_fees + corridor - prem) / V0
            rt_ret = ((prem - proto_fee) - corridor + pos['natural_cap'] * BOND_APY * T_WEEK) / V0

            lp_rets.append(lp_ret)
            rt_rets.append(rt_ret)

        lp_arr = np.array(lp_rets)
        rt_arr = np.array(rt_rets)
        lp_means.append(np.mean(lp_arr) * 100)   # convert to %
        rt_means.append(np.mean(rt_arr) * 100)
        std = np.std(lp_arr, ddof=1) if len(lp_arr) > 1 else 1e-10
        lp_sharpes.append(np.mean(lp_arr) / std if std > 0 else 0)

    ax.plot(alphas, lp_means, 'b-', linewidth=2, label='LP mean return')
    ax.plot(alphas, rt_means, 'r-', linewidth=2, label='RT v2 mean return')
    ax.axhline(0, color='black', ls='-', linewidth=0.5)
    ax.axvline(ALPHA, color='green', ls='--', alpha=0.6, label=f'Current alpha={ALPHA}')

    # Find crossover
    lp_arr_full = np.array(lp_means)
    rt_arr_full = np.array(rt_means)
    both_pos = (lp_arr_full > 0) & (rt_arr_full > 0)
    if np.any(both_pos):
        viable_range = alphas[both_pos]
        ax.axvspan(viable_range[0], viable_range[-1], alpha=0.1, color='green',
                   label=f'Viable: [{viable_range[0]:.2f}, {viable_range[-1]:.2f}]')

    ax.set_title(f'{label}: LP vs RT Mean Return by Alpha')
    ax.set_xlabel('Alpha (upfront fraction)')
    ax.set_ylabel('Mean Weekly Return (% of V0)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Optimal Alpha Sensitivity', fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(DATA_DIR, 'alpha_sensitivity.png'), dpi=150)
plt.close(fig)
print("Saved: data/alpha_sensitivity.png")


## Part VII: Monthly Breakdown

In [ ]:
def monthly_aggregate(results, dates_list, strategy_keys):
    """Aggregate weekly returns into monthly sums (approximate monthly return %)."""
    monthly = {}
    for skey in strategy_keys:
        monthly[skey] = {}
        for ret, dt in zip(results[skey], dates_list):
            month_key = dt.strftime('%Y-%m')
            if month_key not in monthly[skey]:
                monthly[skey][month_key] = 0.0
            monthly[skey][month_key] += ret  # sum of weekly fractional returns
    return monthly

for width_key in ['5pct', '10pct']:
    res = bt_results[width_key]
    meta = bt_meta[width_key]
    label = WIDTHS[width_key]['label']

    strats = ['plain_lp', 'hedged_twopart', 'lp_perp', 'corridor_perp_tail']
    monthly = monthly_aggregate(res, meta['dates'], strats)

    months = sorted(monthly['plain_lp'].keys())
    if not months:
        print(f"No monthly data for {label}")
        continue

    print(f"\n{'=' * 85}")
    print(f"MONTHLY BREAKDOWN -- {label} Width  [cumulative weekly % returns per month]")
    print(f"{'=' * 85}")
    print(f"{'Month':<10} {'Plain LP':>12} {'Hedged (2P)':>12} {'Perp Hedge':>12} {'Corr+Perp':>12} {'Hedge adds':>12}")
    print("-" * 74)

    for m in months:
        plain = monthly['plain_lp'].get(m, 0) * 100
        hedged = monthly['hedged_twopart'].get(m, 0) * 100
        perp = monthly['lp_perp'].get(m, 0) * 100
        cpt = monthly['corridor_perp_tail'].get(m, 0) * 100
        hedge_value = hedged - plain
        print(f"{m:<10} {plain:>+11.2f}% {hedged:>+11.2f}% {perp:>+11.2f}% {cpt:>+11.2f}% {hedge_value:>+11.2f}%"
              f"{'  <-- hedge helps' if hedge_value > 0 else ''}")

    # Plot
    fig, ax = plt.subplots(figsize=(14, 5))
    x = np.arange(len(months))
    plain_vals = [monthly['plain_lp'].get(m, 0) * 100 for m in months]
    hedged_vals = [monthly['hedged_twopart'].get(m, 0) * 100 for m in months]
    perp_vals = [monthly['lp_perp'].get(m, 0) * 100 for m in months]
    cpt_vals = [monthly['corridor_perp_tail'].get(m, 0) * 100 for m in months]

    w = 0.20
    ax.bar(x - 1.5*w, plain_vals, w, label='Plain LP', alpha=0.7, color='blue')
    ax.bar(x - 0.5*w, hedged_vals, w, label='Hedged LP (two-part)', alpha=0.7, color='green')
    ax.bar(x + 0.5*w, perp_vals, w, label='LP + Perp Hedge', alpha=0.7, color='orange')
    ax.bar(x + 1.5*w, cpt_vals, w, label='Corridor + Perp Tail', alpha=0.7, color='brown')
    ax.axhline(0, color='black', ls='-', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(months, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'{label}: Monthly Returns Comparison')
    ax.set_ylabel('Monthly Return (% of V0)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(os.path.join(DATA_DIR, f'monthly_{width_key}.png'), dpi=150)
    plt.close(fig)
    print(f"Saved: data/monthly_{width_key}.png")


## Part IX: Fee Rate Breakeven Analysis

### Question

At what daily yield rate does each hedging strategy become profitable or unprofitable?

This analysis sweeps the daily fee rate from 0.05% to 1.0% per day, reruns the backtest
for each rate, and identifies the **breakeven point** where median weekly return crosses zero.

This answers: "given my pool's fee income, which strategies are viable for me?"

In [ ]:
# ── Fee Rate Breakeven Sweep ──
fee_rates = np.arange(0.0005, 0.0155, 0.0005)  # 0.05% to 1.5%/day

BREAKEVEN_STRATS = ['plain_lp', 'hedged_fixed', 'hedged_jito', 'lp_put', 'lp_put_real', 'lp_perp', 'rt_v1']
STRAT_DISPLAY = {
    'plain_lp': 'Plain LP',
    'hedged_fixed': 'Hedged LP (corridor)',
    'hedged_jito': 'Hedged LP+jito',
    'lp_put': 'LP+Put Spread (BS+IV)',
    'lp_put_real': 'LP+Put (Real Mkt)',
    'lp_perp': 'LP+Perp (80% APY)',
    'rt_v1': 'RT v1 (Pure Insurer)',
}

breakeven_results = {}

for width_key, wcfg in WIDTHS.items():
    w = wcfg['width']
    fs_frac = wcfg['fee_share_bps'] / 10_000
    ref_fee = wcfg['daily_fee']
    sweep = {s: [] for s in BREAKEVEN_STRATS}

    for daily_fee in fee_rates:
        strat_weekly = {s: [] for s in BREAKEVEN_STRATS}
        n_wks = len(close_prices) // 7 - 1

        for wk_i in range(n_wks):
            si = wk_i * 7; ei = si + 7
            if ei >= len(close_prices): break
            S_s = close_prices[si]; S_e = close_prices[ei]

            pos = make_position(S_s, w)
            il = float(cl_value_vec(S_e, pos['L'], pos['p_l'], pos['p_u'])) - pos['V0']

            # In-range fraction
            dp = close_prices[si:ei+1]
            irf = np.mean((dp > pos['p_l']) & (dp < pos['p_u']))
            fees = pos['V0'] * daily_fee * 7 * (irf * 0.95 + 0.05)

            # Trailing vol
            if si >= 30:
                lr = np.diff(np.log(close_prices[si-30:si+1]))
                sl = float(np.std(lr, ddof=1) * np.sqrt(365))
            else:
                sl = 0.65

            fv = fv_quadrature(pos, sl)
            pay = float(corridor_payoff_vec(
                np.array([S_e]), S_s, pos['barrier'], pos['natural_cap'],
                pos['L'], pos['p_l'], pos['p_u'])[0])
            prem = fv * MARKUP
            eff_prem = max(0, prem - fees * fs_frac)

            # Put spread
            ps_cost = (pos['V0']/S_s) * (bs_put(S_s, S_s, sl, T_WEEK) - bs_put(S_s, pos['barrier'], sl, T_WEEK))
            ps_pay = (pos['V0']/S_s) * max(0, max(0, S_s-S_e) - max(0, pos['barrier']-S_e))

            # Perp (80%)
            delta = cl_delta(S_s, pos['L'], pos['p_l'], pos['p_u'])
            perp_pnl = -delta * (S_e - S_s)
            perp_fund = abs(delta) * S_s * PERP_FUNDING_APY * T_WEEK
            perp_fee = abs(delta) * S_s * 2 * PERP_FEE_BPS / 10_000

            jito_ret = pos['V0'] * SOL_FRACTION * ((1+0.07)**(1/52)-1)

            V0 = pos['V0']
            strat_weekly['plain_lp'].append((il + fees) / V0)
            strat_weekly['hedged_fixed'].append((il + fees + pay - eff_prem) / V0)
            strat_weekly['hedged_jito'].append((il + fees + pay - eff_prem + jito_ret) / V0)
            strat_weekly['lp_put'].append((il + fees + ps_pay - ps_cost) / V0)
            strat_weekly['lp_perp'].append((il + fees + perp_pnl - perp_fund - perp_fee) / V0)

            # LP + Put Spread (real market)
            real_ps = REAL_PUT_SPREAD.get(width_key)
            if real_ps and real_ps.get('cost_per_sol'):
                rps_cost = real_ps['cost_per_sol'] * (V0 / S_s)
                dte_r = 7.0 / max(real_ps['info']['dte'], 1)
                rps_cost *= min(dte_r, 2.0)
                rps_pay = (V0 / S_s) * max(0, max(0, S_s - S_e) - max(0, pos['barrier'] - S_e))
                strat_weekly['lp_put_real'].append((il + fees + rps_pay - rps_cost - TX_COST_SOL * S_s * 2) / V0)
            else:
                rps_cost = ps_cost * 1.5
                strat_weekly['lp_put_real'].append((il + fees + ps_pay - rps_cost - TX_COST_SOL * S_s * 2) / V0)

            # RT v1 (Pure Insurer): receives two-part premium, pays corridor claims
            wf_e = V0 * daily_fee * 7 * (irf * 0.95 + 0.05)
            beta_sw = max(0, (MARKUP - ALPHA) * fv / max(wf_e, 1e-10))
            s7_sw = float(np.std(np.diff(np.log(close_prices[max(0,si-7):si+1])), ddof=1) * np.sqrt(365)) if si >= 7 else sl
            vi_sw = np.clip(s7_sw / max(sl, 1e-6), 0.5, 2.0)
            prem_tp_sw = ALPHA * fv * vi_sw + beta_sw * wf_e
            proto_sw = prem_tp_sw * PROTOCOL_FEE_BPS / BPS
            strat_weekly['rt_v1'].append(((prem_tp_sw - proto_sw) - pay) / V0)

        for s in BREAKEVEN_STRATS:
            arr = np.array(strat_weekly[s])
            sh = float(np.mean(arr)/np.std(arr)) if np.std(arr) > 1e-10 else 0
            sweep[s].append(dict(fee=daily_fee, median=float(np.median(arr)*100),
                                  mean=float(np.mean(arr)*100), sharpe=sh,
                                  p_pos=float(np.mean(arr > 0)*100)))

    breakeven_results[width_key] = sweep
    print(f'{width_key} sweep done: {len(fee_rates)} rates x {n_wks} weeks')

print('Breakeven sweep complete.')
# ── Adaptive Markup Breakeven Sweep (regime_switching x markup floors) ──
print("\nRunning adaptive markup breakeven sweep...")
adaptive_breakeven_results = {}
rs_fn = IV_RV_SCENARIOS['regime_switching']

for width_key, wcfg in WIDTHS.items():
    w = wcfg['width']
    fs_frac = wcfg['fee_share_bps'] / 10_000
    adaptive_breakeven_results[width_key] = {}

    for mfloor in MARKUP_FLOOR_VALUES:
        sweep = {s: [] for s in BREAKEVEN_STRATS}
        n_wks = len(close_prices) // 7 - 1

        for daily_fee in fee_rates:
            strat_weekly = {s: [] for s in BREAKEVEN_STRATS}

            for wk_i in range(n_wks):
                si = wk_i * 7; ei = si + 7
                if ei >= len(close_prices): break
                S_s = close_prices[si]; S_e = close_prices[ei]

                pos = make_position(S_s, w)
                il = float(cl_value_vec(S_e, pos['L'], pos['p_l'], pos['p_u'])) - pos['V0']
                dp = close_prices[si:ei+1]
                irf = np.mean((dp > pos['p_l']) & (dp < pos['p_u']))
                fees = pos['V0'] * daily_fee * 7 * (irf * 0.95 + 0.05)

                if si >= 30:
                    lr = np.diff(np.log(close_prices[si-30:si+1]))
                    sl = float(np.std(lr, ddof=1) * np.sqrt(365))
                else:
                    sl = 0.65
                if si >= 7:
                    lr7 = np.diff(np.log(close_prices[max(0,si-7):si+1]))
                    s7_loc = float(np.std(lr7, ddof=1) * np.sqrt(365))
                else:
                    s7_loc = sl

                fv = fv_quadrature(pos, sl)
                pay = float(corridor_payoff_vec(
                    np.array([S_e]), S_s, pos['barrier'], pos['natural_cap'],
                    pos['L'], pos['p_l'], pos['p_u'])[0])

                # Adaptive markup
                stress_proxy = (s7_loc / max(sl, 1e-6) > 1.5)
                eff_mk, _ = compute_adaptive_markup_for_week(s7_loc, sl, stress_proxy, rs_fn, mfloor)

                prem = fv * eff_mk
                eff_prem = max(0, prem - fees * fs_frac)

                ps_cost = (pos['V0']/S_s) * (bs_put(S_s, S_s, sl, T_WEEK) - bs_put(S_s, pos['barrier'], sl, T_WEEK))
                ps_pay = (pos['V0']/S_s) * max(0, max(0, S_s-S_e) - max(0, pos['barrier']-S_e))

                delta = cl_delta(S_s, pos['L'], pos['p_l'], pos['p_u'])
                perp_pnl = -delta * (S_e - S_s)
                perp_fund = abs(delta) * S_s * PERP_FUNDING_APY * T_WEEK
                perp_fee = abs(delta) * S_s * 2 * PERP_FEE_BPS / 10_000

                jito_ret = pos['V0'] * SOL_FRACTION * ((1+0.07)**(1/52)-1)

                V0 = pos['V0']
                strat_weekly['plain_lp'].append((il + fees) / V0)
                strat_weekly['hedged_fixed'].append((il + fees + pay - eff_prem) / V0)
                strat_weekly['hedged_jito'].append((il + fees + pay - eff_prem + jito_ret) / V0)
                strat_weekly['lp_put'].append((il + fees + ps_pay - ps_cost) / V0)
                strat_weekly['lp_perp'].append((il + fees + perp_pnl - perp_fund - perp_fee) / V0)

                real_ps = REAL_PUT_SPREAD.get(width_key)
                if real_ps and real_ps.get('cost_per_sol'):
                    rps_cost = real_ps['cost_per_sol'] * (V0 / S_s)
                    dte_r = 7.0 / max(real_ps['info']['dte'], 1)
                    rps_cost *= min(dte_r, 2.0)
                    rps_pay = (V0 / S_s) * max(0, max(0, S_s - S_e) - max(0, pos['barrier'] - S_e))
                    strat_weekly['lp_put_real'].append((il + fees + rps_pay - rps_cost - TX_COST_SOL * S_s * 2) / V0)
                else:
                    rps_cost = ps_cost * 1.5
                    strat_weekly['lp_put_real'].append((il + fees + ps_pay - rps_cost - TX_COST_SOL * S_s * 2) / V0)

                wf_e = V0 * daily_fee * 7 * (irf * 0.95 + 0.05)
                beta_sw = max(0, (eff_mk - ALPHA) * fv / max(wf_e, 1e-10))
                vi_sw = np.clip(s7_loc / max(sl, 1e-6), 0.5, 2.0)
                prem_tp_sw = ALPHA * fv * vi_sw + beta_sw * wf_e
                proto_sw = prem_tp_sw * PROTOCOL_FEE_BPS / BPS
                strat_weekly['rt_v1'].append(((prem_tp_sw - proto_sw) - pay) / V0)

            for s in BREAKEVEN_STRATS:
                arr = np.array(strat_weekly[s])
                sh = float(np.mean(arr)/np.std(arr)) if np.std(arr) > 1e-10 else 0
                sweep[s].append(dict(fee=daily_fee, median=float(np.median(arr)*100),
                                      mean=float(np.mean(arr)*100), sharpe=sh,
                                      p_pos=float(np.mean(arr > 0)*100)))

        adaptive_breakeven_results[width_key][mfloor] = sweep
        print(f'  {width_key} floor={mfloor}: sweep done')

print('Adaptive breakeven sweep complete.')


In [ ]:
# ── Breakeven table ──
for width_key, wcfg in WIDTHS.items():
    sweep = breakeven_results[width_key]
    ref_fee = wcfg['daily_fee']
    print(f'\n{"="*80}')
    print(f'{wcfg["label"]} WIDTH -- BREAKEVEN FEE RATES')
    print(f'{"="*80}')
    print(f'{"Strategy":<30} {"Breakeven":>12} {"At ref fee":>12} {"Status":>12}')
    print(f'{"-"*70}')

    for s in BREAKEVEN_STRATS:
        medians = np.array([r['median'] for r in sweep[s]])
        fees_arr = np.array([r['fee'] for r in sweep[s]])
        breakeven = None
        for j in range(len(medians)-1):
            if medians[j] <= 0 and medians[j+1] > 0:
                frac = -medians[j] / (medians[j+1] - medians[j])
                breakeven = fees_arr[j] + frac * (fees_arr[j+1] - fees_arr[j])
                break
        if breakeven is None and medians[0] > 0: breakeven = 0
        if breakeven is None: breakeven = float('inf')
        ref_idx = int(np.argmin(np.abs(fees_arr - ref_fee)))
        ref_med = sweep[s][ref_idx]['median']
        if breakeven == 0: be_str = '< 0.05%/d'
        elif breakeven == float('inf'): be_str = '> 1.5%/d'
        else: be_str = f'{breakeven*100:.2f}%/d'
        status = 'PROFITABLE' if ref_med > 0 else 'UNPROFITABLE'
        print(f'{STRAT_DISPLAY[s]:<30} {be_str:>12} {ref_med:>+11.3f}% {status:>12}')

In [ ]:
# ── Profitability curves plot ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors_be = {'plain_lp': 'steelblue', 'hedged_fixed': 'seagreen',
             'hedged_jito': 'darkgreen', 'lp_put': 'brown', 'lp_put_real': 'red',
             'lp_perp': 'teal', 'rt_v1': 'purple'}

for col, (width_key, wcfg) in enumerate(WIDTHS.items()):
    sweep = breakeven_results[width_key]
    ref_fee = wcfg['daily_fee']

    ax = axes[0, col]
    for s in BREAKEVEN_STRATS:
        x_vals = [r['fee']*100 for r in sweep[s]]
        y_vals = [r['median'] for r in sweep[s]]
        ax.plot(x_vals, y_vals, '-', color=colors_be[s], lw=2, label=STRAT_DISPLAY[s])
    ax.axhline(y=0, color='black', lw=1, ls='--')
    ax.axvline(x=ref_fee*100, color='gray', lw=1.5, ls=':', label=f'Ref {ref_fee*100:.2f}%')
    ax.set_xlabel('Daily Fee Rate (%)'); ax.set_ylabel('Median Weekly Return (%)')
    ax.set_title(f'{wcfg["label"]} -- Median Return vs Fee Rate')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    ax = axes[1, col]
    for s in BREAKEVEN_STRATS:
        x_vals = [r['fee']*100 for r in sweep[s]]
        y_vals = [r['sharpe'] for r in sweep[s]]
        ax.plot(x_vals, y_vals, '-', color=colors_be[s], lw=2, label=STRAT_DISPLAY[s])
    ax.axhline(y=0, color='black', lw=1, ls='--')
    ax.axvline(x=ref_fee*100, color='gray', lw=1.5, ls=':', label=f'Ref {ref_fee*100:.2f}%')
    ax.set_xlabel('Daily Fee Rate (%)'); ax.set_ylabel('Sharpe Ratio')
    ax.set_title(f'{wcfg["label"]} -- Sharpe vs Fee Rate')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.suptitle('Strategy Profitability by Fee Rate (real data, markup=1.10x)', fontsize=13, y=1.01)
plt.tight_layout()
os.makedirs('data', exist_ok=True)
plt.savefig('data/breakeven_fee_rates.png', dpi=150, bbox_inches='tight'); plt.close()
print('Saved: data/breakeven_fee_rates.png')

### Interpretation

The breakeven fee rate is the daily yield at which each strategy's median weekly return
crosses zero on real SOL/USDC data.

- **Below breakeven**: the strategy loses money in the typical week
- **Above breakeven**: the strategy is profitable in the typical week

The corridor certificate has a higher breakeven than Plain LP (because the premium is an
additional cost), but **above that breakeven it delivers far better risk-adjusted returns**
(higher Sharpe, lower drawdown). The LP pays the premium to buy Sharpe ratio, not raw return.

The Sharpe plot (bottom row) shows that the corridor overtakes the perp hedge at moderate
fee rates and the gap widens as fees increase — confirming the corridor's structural advantage
for actively-traded pools.

## Part VIII: Conclusions

### 8.1 Summary Table

In [ ]:
print("=" * 120)
print("FINAL SUMMARY: ANNUALIZED MEDIAN RETURNS AND KEY METRICS")
print("=" * 120)
print()

for width_key in ['5pct', '10pct']:
    res = bt_results[width_key]
    meta = bt_meta[width_key]
    label = WIDTHS[width_key]['label']

    print(f"--- {label} Width ({meta['n_weeks']} weeks) ---")
    print(f"{'Strategy':<32} {'Ann. Median':>12} {'Sharpe':>8} {'MaxDD':>10} {'P(+)':>7}")
    print("-" * 75)

    for skey, slabel in STRATEGY_LABELS.items():
        st = compute_stats(res[skey])
        if not st:
            continue
        print(f"{slabel:<32} {st['ann_median']*100:>+10.2f}% {st['sharpe']:>7.3f} "
              f"{st['max_dd']*100:>+9.2f}% {st['p_pos']:>6.1%}")
    print()


### 8.2 Recommended Parameters

In [ ]:
print("=== RECOMMENDED PARAMETERS (v2) ===")
print()
print("Protocol-level:")
print(f"  MARKUP          = {MARKUP}x (down from 1.20x in v1)")
print(f"  ALPHA           = {ALPHA} (40% upfront)")
print(f"  U_MAX           = {U_MAX_BPS / BPS:.0%}")
print(f"  PROTOCOL_FEE    = {PROTOCOL_FEE_BPS / BPS:.1%}")
print()
print("Width-specific:")
for key, wcfg in WIDTHS.items():
    print(f"  {wcfg['label']}:")
    print(f"    daily_fee_rate = {wcfg['daily_fee']:.4f}")
    print(f"    severity_ppm   = {wcfg['severity_ppm']:,}")
    print(f"    barrier_pct    = {wcfg['barrier_pct']}")
    print(f"    beta           = {wcfg.get('_beta', 'N/A')}")
print()
print("Dropped: +-15% width (unviable -- insufficient fee generation relative to risk)")


### 8.3 When to Use Fixed vs Two-Part Premium

In [ ]:
print("=== FIXED vs TWO-PART PREMIUM: DECISION GUIDE ===")
print()

for width_key in ['5pct', '10pct']:
    res = bt_results[width_key]
    label = WIDTHS[width_key]['label']

    fixed_arr = np.array(res['hedged_fixed'])
    tp_arr = np.array(res['hedged_twopart'])
    n = len(fixed_arr)

    tp_better = np.sum(tp_arr > fixed_arr)
    fixed_better = np.sum(fixed_arr > tp_arr)

    tp_stats = compute_stats(res['hedged_twopart'])
    fixed_stats = compute_stats(res['hedged_fixed'])

    print(f"--- {label} ---")
    print(f"  Two-part wins {tp_better}/{n} weeks ({tp_better/n:.0%})")
    print(f"  Fixed wins    {fixed_better}/{n} weeks ({fixed_better/n:.0%})")
    print(f"  Two-part Sharpe: {tp_stats['sharpe']:.3f},  Fixed Sharpe: {fixed_stats['sharpe']:.3f}")
    print(f"  Two-part MaxDD:  {tp_stats['max_dd']*100:+.2f}%,  Fixed MaxDD:  {fixed_stats['max_dd']*100:+.2f}%")
    print()

print("Recommendation:")
print("  - Two-part premium is preferred in normal/low-vol environments")
print("  - Fixed premium is simpler and may be preferred when vol_indicator is noisy")
print("  - For protocol launch: default to two-part, allow LP to choose fixed as option")


### 8.4 Comparison with Alternative Hedges

In [ ]:
print("=== CORRIDOR CERTIFICATE vs ALTERNATIVE HEDGES ===")
print()

for width_key in ['5pct', '10pct']:
    res = bt_results[width_key]
    label = WIDTHS[width_key]['label']

    tp_stats = compute_stats(res['hedged_twopart'])
    put_stats = compute_stats(res['lp_put_spread'])
    perp_stats = compute_stats(res['lp_perp'])
    cpt_stats = compute_stats(res['corridor_perp_tail'])
    plain_stats = compute_stats(res['plain_lp'])

    print(f"--- {label} ---")
    print(f"{'Metric':<20} {'Corridor (2P)':>14} {'Put Spread':>14} {'Perp Hedge':>14} {'Corr+Perp':>14} {'No Hedge':>14}")
    print("-" * 95)

    metrics = [
        ('Ann. Median', 'ann_median', '{:+.2%}'),
        ('Sharpe', 'sharpe', '{:.3f}'),
        ('Max DD', 'max_dd', '{:+.2%}'),
        ('P(positive)', 'p_pos', '{:.1%}'),
        ('5th percentile', 'pct5', '{:+.3%}'),
    ]
    all_stats = [tp_stats, put_stats, perp_stats, cpt_stats, plain_stats]
    for mname, mkey, fmt in metrics:
        vals = [fmt.format(s[mkey]) for s in all_stats]
        print(f"{mname:<20} {vals[0]:>14} {vals[1]:>14} {vals[2]:>14} {vals[3]:>14} {vals[4]:>14}")
    print()

print("Key takeaways:")
print("  - Corridor certificate: bounded cost, no margin, no liquidation risk")
print("  - Put spread: similar payoff but more expensive (IV premium + bid-ask spread)")
print("  - Perp delta hedge: continuous rebalancing needed, funding costs erode returns")
print("  - Corridor + Perp Tail: hybrid approach hedges both in-range IL and below-barrier drops")
print("  - The corridor cert is the only hedge that ties directly to the CL value function")


---

**End of Analysis**

All plots saved to `data/` directory. Key files:
- `data/volatility_rolling.png` -- Realized volatility over time
- `data/price_and_volume.png` -- SOL/USDC price and volume
- `data/fv_vs_heuristic.png` -- Fair value vs heuristic across vol
- `data/premium_decomposition.png` -- Two-part premium components
- `data/strategy_comparison_5pct.png` -- Strategy comparison (+-5%)
- `data/strategy_comparison_10pct.png` -- Strategy comparison (+-10%)
- `data/alpha_sensitivity.png` -- Optimal alpha sweep
- `data/monthly_breakdown_5pct.png` -- Monthly returns (+-5%)
- `data/monthly_breakdown_10pct.png` -- Monthly returns (+-10%)